# Training Faster R-CNN with k-fold cross validation

The code in here is roughly based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Logging code

In [ ]:
import csv
from datetime import datetime


def setup_csv(file_name: str):
    """Set up the CSV file with the appropriate headers."""
    with open(file_name, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(
            [
                "stage",
                "fold",
                "epoch",
                "train_loss",
                "train_loss_classifier",
                "train_loss_box_reg",
                "train_loss_objectness",
                "train_loss_rpn_box_reg",
                "val_loss",
                "val_loss_classifier",
                "val_loss_box_reg",
                "val_loss_objectness",
                "val_loss_rpn_box_reg",
                "val_avg_precision_iou_50",
                "val_avg_recall_iou_50",
                "val_mae",
                "val_smape",
            ]
        )

    def append_to_csv(row):
        """Append a row of data to the CSV file."""
        with open(file_name, "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(row)

    return append_to_csv


start_time_str = datetime.now().strftime("%Y%m%d_%H%M")

# Hyperparameters

In [ ]:
BATCH_SIZE = 4

## Training code


In [1]:
from engine import evaluate, train_one_epoch
from torch.optim import AdamW
import torch
import utils
from agar_dataset import AgarBoundingBoxDataset
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2


def create_fasterrcnn_model(num_classes: int, pretrained=True):
    model = fasterrcnn_resnet50_fpn_v2(
        weights="DEFAULT" if pretrained else None,
        num_classes=num_classes,
        box_detections_per_img=300,
    )

    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")


def train_model(
    stage: str,
    train_set: torch.utils.data.Dataset,
    val_set: torch.utils.data.Dataset = None,
    num_epochs=20,
    model=None,
    save_scores: callable = None,
    fold: int = None,
):
    should_val = val_set is not None

    train_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        num_workers=4,
        shuffle=True,
        collate_fn=utils.collate_fn,
    )

    if model is None:
        model = create_fasterrcnn_model(AgarBoundingBoxDataset.num_classes).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=0.001, weight_decay=0.01)

    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        train_losses = train_one_epoch(
            model, optimizer, train_loader, device, epoch, print_freq=10
        )

        avg_train_loss_classifier = sum(
            [loss["loss_classifier"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_box_reg = sum(
            [loss["loss_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_objectness = sum(
            [loss["loss_objectness"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss_rpn_box_reg = sum(
            [loss["loss_rpn_box_reg"] for loss in train_losses]
        ) / len(train_losses)
        avg_train_loss = sum(
            [
                (
                    loss["loss_classifier"]
                    + loss["loss_box_reg"]
                    + loss["loss_objectness"]
                    + loss["loss_rpn_box_reg"]
                )
                for loss in train_losses
            ]
        ) / len(train_losses)

        # check loss on validation set
        if should_val:
            val_loader = torch.utils.data.DataLoader(
                val_set,
                batch_size=BATCH_SIZE,
                num_workers=4,
                shuffle=False,
                collate_fn=utils.collate_fn,
            )

            with torch.no_grad():
                val_losses = train_one_epoch(
                    model,
                    optimizer,
                    val_loader,
                    device,
                    epoch,
                    print_freq=10,
                    optimize=False,
                )

                avg_val_loss_classifier = sum(
                    [loss["loss_classifier"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_box_reg = sum(
                    [loss["loss_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_objectness = sum(
                    [loss["loss_objectness"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss_rpn_box_reg = sum(
                    [loss["loss_rpn_box_reg"] for loss in val_losses]
                ) / len(val_losses)
                avg_val_loss = sum(
                    [
                        (
                            loss["loss_classifier"]
                            + loss["loss_box_reg"]
                            + loss["loss_objectness"]
                            + loss["loss_rpn_box_reg"]
                        )
                        for loss in val_losses
                    ]
                ) / len(val_losses)

                # evaluate on the validation dataset
                coco_evaluator, mae, smape = evaluate(model, val_loader, device=device)

                stats = coco_evaluator.coco_eval["bbox"].stats

                avg_precision_iou_50 = stats[0]
                avg_recall_iou_50 = stats[8]

        if save_scores is not None:
            save_scores(
                [
                    stage,
                    float(fold) if fold is not None else "N/A",
                    float(epoch),
                    float(avg_train_loss),
                    float(avg_train_loss_classifier),
                    float(avg_train_loss_box_reg),
                    float(avg_train_loss_objectness),
                    float(avg_train_loss_rpn_box_reg),
                    float(avg_val_loss) if should_val else "N/A",
                    float(avg_val_loss_classifier) if should_val else "N/A",
                    float(avg_val_loss_box_reg) if should_val else "N/A",
                    float(avg_val_loss_objectness) if should_val else "N/A",
                    float(avg_val_loss_rpn_box_reg) if should_val else "N/A",
                    float(avg_precision_iou_50) if should_val else "N/A",
                    float(avg_recall_iou_50) if should_val else "N/A",
                    float(mae) if should_val else "N/A",
                    float(smape) if should_val else "N/A",
                ],
            )

    return model

Using cuda for training.


## Training on the AGAR dataset

Simple training and validation run on the high-resolution images of the AGAR
dataset.


In [2]:
import json
from dataset_transforms import get_transforms

# num_epochs_agar = 20

# print("Training on AGAR dataset\n" + "-" * 10 + "\n")

# # These box data for these IDs of the AGAR dataset have negative x values for the bounding boxes.
# faulty_ids = ["5794", "11988"]

# with open("../../AGAR_dataset/training_lists/higher_resolution_train.txt") as f:
#     train_list = json.load(f)
#     train_list = [int(id) for id in train_list if id.isdigit() and id not in faulty_ids]

# with open("../../AGAR_dataset/training_lists/higher_resolution_val.txt") as f:
#     val_list = json.load(f)
#     val_list = [int(id) for id in val_list if id.isdigit() and id not in faulty_ids]

# print(f"Training on {len(train_list)} images.")
# print(f"Validating on {len(val_list)} images.")

# train_dataset = AgarBoundingBoxDataset(
#     "../../AGAR_dataset/dataset", train_list, get_transforms(is_train=True)
# )
# val_dataset = AgarBoundingBoxDataset(
#     "../../AGAR_dataset/dataset", val_list, get_transforms(is_train=False)
# )

write_row = setup_csv(f"rrrandom-training_run_{start_time_str}.csv")

# agar_trained_model = train_model(
#     "agar_pretraining",
#     train_set=train_dataset,
#     val_set=val_dataset,
#     num_epochs=num_epochs_agar,
#     save_scores=write_row,
# )

# agar_model_filename = f"agar_model_{start_time_str}.pth"

# torch.save(agar_trained_model.state_dict(), agar_model_filename)

## K-fold cross validation on own dataset

K different models are trained and validated here


In [3]:
from instance_dataset import BoundingBoxDataset


# def pretrained_agar_model():
#     model = get_model_instance_segmentation(BoundingBoxDataset.num_classes).to(device)
#     model.load_state_dict(torch.load(agar_model_filename))
#     return model

In [4]:
from sklearn.model_selection import KFold


k_folds = 5
num_epochs = 200
kfold = KFold(n_splits=k_folds, shuffle=True)

# use our dataset and defined transformations
full_dataset = BoundingBoxDataset("../../training_data", get_transforms(is_train=True))

for fold, (train_ids, val_ids) in enumerate(kfold.split(full_dataset)):
    print(f"Training fold {fold}\n" + "-" * 10 + "\n")

    train_dataset = torch.utils.data.Subset(full_dataset, train_ids)
    train_dataset.dataset.transforms = get_transforms(is_train=True, rotate=True)

    val_dataset = torch.utils.data.Subset(full_dataset, val_ids)
    val_dataset.dataset.transforms = get_transforms(is_train=False)

    base_model = create_fasterrcnn_model(BoundingBoxDataset.num_classes).to(device)

    fold_model = train_model(
        "kfold",
        train_set=train_dataset,
        val_set=val_dataset,
        num_epochs=num_epochs,
        model=base_model,
        save_scores=write_row,
        fold=fold,
    )

    torch.save(fold_model.state_dict(), f"rrrandom-model_fold_{fold}.pth")

Training fold 0
----------



/home/ms/l/ld580269/miniconda3/envs/bcs/lib/python3.12/site-packages/torchvision/models/detection/backbone_utils.py:162: UserWarning: Changing trainable_backbone_layers has no effect if neither pretrained nor pretrained_backbone have been set to True, falling back to trainable_backbone_layers=5 so that all layers are trainable
  warnings.warn(


Epoch: [0]  [ 0/12]  eta: 0:00:51  lr: 0.000092  loss: 1.6802 (1.6802)  loss_classifier: 0.6476 (0.6476)  loss_box_reg: 0.0462 (0.0462)  loss_objectness: 0.6963 (0.6963)  loss_rpn_box_reg: 0.2902 (0.2902)  time: 4.2696  data: 1.8016  max mem: 14231
Epoch: [0]  [10/12]  eta: 0:00:03  lr: 0.001000  loss: 1.6802 (1.8218)  loss_classifier: 0.4058 (0.4518)  loss_box_reg: 0.0948 (0.0897)  loss_objectness: 0.6771 (0.6664)  loss_rpn_box_reg: 0.5502 (0.6139)  time: 1.5859  data: 0.2795  max mem: 14442
Epoch: [0]  [11/12]  eta: 0:00:01  lr: 0.001000  loss: 1.6729 (1.8001)  loss_classifier: 0.3473 (0.4430)  loss_box_reg: 0.0948 (0.0925)  loss_objectness: 0.6690 (0.6622)  loss_rpn_box_reg: 0.4866 (0.6024)  time: 1.5638  data: 0.2669  max mem: 14442
Epoch: [0] Total time: 0:00:18 (1.5673 s / it)
Epoch: [0]  [0/3]  eta: 0:00:06  lr: 0.000001  loss: 1.3083 (1.3083)  loss_classifier: 0.3282 (0.3282)  loss_box_reg: 0.2522 (0.2522)  loss_objectness: 0.5034 (0.5034)  loss_rpn_box_reg: 0.2245 (0.2245)  ti

KeyboardInterrupt: 

## Final training run

The final training run on all of the available training data.


In [ ]:
print(f"Final training run\n" + "-" * 10 + "\n")

final_model = train_model(
    "final_training",
    train_set=full_dataset,
    model=create_fasterrcnn_model(BoundingBoxDataset.num_classes).to(device),
    num_epochs=num_epochs,
    save_scores=write_row,
)

# save the model
torch.save(
    final_model.state_dict(),
    f"rrrandom-faster-rcnn-{start_time_str}.pth",
)

## Evaluation

Evaluate the final model on key metrics.


In [ ]:
print(f"Evaluating on test set\n" + "-" * 10 + "\n")

test_dataset = BoundingBoxDataset("../../test_data", get_transforms(is_train=False))

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=utils.collate_fn
)

coco_evaluator, mae, smape = evaluate(final_model, test_loader, device=device)

stats = coco_evaluator.coco_eval["bbox"].stats.tolist()

with open(f"rrrandom-test_results_{start_time_str}.txt", "w") as f:
    f.write(f"Average absolute error: {mae}\n")
    f.write(f"Average relative error: {smape}\n")

    f.write(json.dumps(stats, indent=4))